# Seasonal Agriculture Performance Analysis

## Major Data Analytics Project

**Domain:** Agriculture  
**Project Theme:** Seasonal Agriculture Performance  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

This notebook is prepared for Google Colab. Upload the CSV when asked, or keep it in `/content` with the same filename.


## 1. Problem Statement

Agricultural performance changes across seasons because rainfall, temperature, soil condition, irrigation practice, input use and market outcome vary over time. The problem is to analyze the given farm-level dataset and identify how yield, production, profit, water usage and crop health risk differ across Kharif, Rabi and Zaid seasons.


## 2. Project Objectives

- Understand the dataset structure and quality.
- Rename features inside the notebook for easier analysis.
- Clean missing values and duplicate records in a working copy.
- Explore seasonal patterns using statistics and visualizations.
- Compare yield, production, profit, water use and risk across seasons.
- Identify useful insights, recommendations and limitations.


## 3. Dataset

The dataset contains farm records with crop, season, location, weather, soil, fertilizer, irrigation, yield, production, cost, revenue, profit, water use and disease/pest risk details.

**Dataset file:** `seasonal_agriculture_performance_dataset (3).csv`


## 4. Initial Data Understanding


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


In [ ]:
# Fresh label: Source file reading for Google Colab
csv_file_name = "seasonal_agriculture_performance_dataset (3).csv"
source_file = Path("/content") / csv_file_name

if not source_file.exists():
    print("CSV file not found in /content. Please upload the dataset file now.")
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            csv_file_name = next(iter(uploaded))
            source_file = Path("/content") / csv_file_name
    except ModuleNotFoundError:
        print("If you are not using Colab, place the CSV in the same folder as this notebook.")
        source_file = Path(csv_file_name)

raw_data = pd.read_csv(source_file)
print("Dataset opened successfully for analysis:", source_file)


In [ ]:
# Fresh label: Dataset size check
print("Farm records:", raw_data.shape[0])
print("Available features:", raw_data.shape[1])


In [ ]:
# Fresh label: First record preview
raw_data.head()


In [ ]:
# Fresh label: Last record preview
raw_data.tail()


In [ ]:
# Fresh label: Random farm sample
raw_data.sample(5, random_state=42)


In [ ]:
# Fresh label: Available data fields
raw_data.columns.tolist()


In [ ]:
# Fresh label: Field type and completeness scan
raw_data.info()


In [ ]:
# Fresh label: Numeric profile overview
raw_data.describe()


In [ ]:
# Fresh label: Text/category profile overview
raw_data.describe(include="object")


In [ ]:
# Fresh label: Working feature aliases used only inside this notebook
feature_name_map = {
    "Farm_ID": "farm_code",
    "State": "state_name",
    "District": "district_name",
    "Crop": "crop_type",
    "Season": "growing_season",
    "Farm_Area_Hectares": "field_area_ha",
    "Rainfall_mm": "rainfall_level_mm",
    "Avg_Temperature_C": "mean_temp_c",
    "Humidity_pct": "relative_humidity_pct",
    "Sunlight_Hours_Day": "daily_sunlight_hours",
    "Soil_pH": "soil_acidity_ph",
    "Soil_Moisture_pct": "soil_water_content_pct",
    "Nitrogen_kg_ha": "nitrogen_input_kg_ha",
    "Phosphorus_kg_ha": "phosphorus_input_kg_ha",
    "Potassium_kg_ha": "potassium_input_kg_ha",
    "Irrigation_Method": "watering_system",
    "Fertilizer_kg_ha": "fertilizer_dose_kg_ha",
    "Pesticide_Litre_ha": "pesticide_use_litre_ha",
    "Seed_Quality_Score": "seed_grade_score",
    "Yield_Tonnes_Ha": "harvest_yield_t_ha",
    "Production_Tonnes": "total_harvest_tonnes",
    "Market_Price_INR_Tonne": "selling_price_inr_tonne",
    "Total_Cost_INR": "cultivation_cost_inr",
    "Revenue_INR": "gross_income_inr",
    "Profit_INR": "net_return_inr",
    "Water_Used_m3": "irrigation_volume_m3",
    "Water_Efficiency_t_per_1000m3": "water_productivity_t_1000m3",
    "Disease_Pest_Risk_pct": "crop_health_risk_pct"
}

farm = raw_data.rename(columns=feature_name_map).copy()
pd.DataFrame({"Original column": list(feature_name_map.keys()), "Notebook alias": list(feature_name_map.values())})


## 5. Data Quality Analysis

Missing values, duplicate rows and category values are checked before analysis.


In [ ]:
# Fresh label: Blank value audit
missing_count = farm.isna().sum().sort_values(ascending=False)
missing_percent = (farm.isna().mean() * 100).sort_values(ascending=False).round(2)
missing_report = pd.DataFrame({"Blank Rows": missing_count, "Blank Percent": missing_percent})
missing_report[missing_report["Blank Rows"] > 0]


In [ ]:
# Fresh label: Blank value visual check
plt.figure(figsize=(12, 5))
missing_plot = missing_count[missing_count > 0]
if len(missing_plot) > 0:
    sns.barplot(x=missing_plot.index, y=missing_plot.values)
    plt.xticks(rotation=60, ha="right")
    plt.title("Blank Values By Notebook Alias")
    plt.xlabel("Feature alias")
    plt.ylabel("Blank rows")
    plt.tight_layout()
    plt.show()
else:
    print("No blank values found.")


In [ ]:
# Fresh label: Repeated row audit
print("Repeated farm records:", farm.duplicated().sum())


In [ ]:
# Fresh label: Working-copy duplicate handling
analysis_data = farm.drop_duplicates().reset_index(drop=True).copy()
print("Working-copy shape after repeated-row check:", analysis_data.shape)


In [ ]:
# Fresh label: Category variety scan
category_features = analysis_data.select_dtypes(include="object").columns.tolist()
for col in category_features:
    print(f"\n{col}:")
    print("Unique labels:", analysis_data[col].nunique())
    print(analysis_data[col].dropna().unique()[:20])


### Missing Value Treatment

Numerical missing values are filled with median values. Categorical missing values are filled with `Unknown`. This treatment happens only in the notebook working copy.


In [ ]:
# Fresh label: Notebook-only blank value treatment
for col in analysis_data.select_dtypes(include=np.number).columns:
    if analysis_data[col].isna().sum() > 0:
        analysis_data[col] = analysis_data[col].fillna(analysis_data[col].median())
for col in analysis_data.select_dtypes(include="object").columns:
    if analysis_data[col].isna().sum() > 0:
        analysis_data[col] = analysis_data[col].fillna("Unknown")
print("Remaining blank values in working copy:", analysis_data.isna().sum().sum())


## 6. Feature / Variable Review


In [ ]:
# Fresh label: Numeric and category alias split
numeric_features = analysis_data.select_dtypes(include=np.number).columns.tolist()
category_features = analysis_data.select_dtypes(include="object").columns.tolist()
print("Numeric aliases:")
print(numeric_features)
print("\nCategory aliases:")
print(category_features)


In [ ]:
# Fresh label: Concept-based field grouping
variable_groups = {
    "Identifier variables": ["farm_code"],
    "Categorical variables": ["state_name", "district_name", "crop_type", "growing_season", "watering_system"],
    "Environmental variables": ["rainfall_level_mm", "mean_temp_c", "relative_humidity_pct", "daily_sunlight_hours", "soil_acidity_ph", "soil_water_content_pct"],
    "Operational variables": ["field_area_ha", "nitrogen_input_kg_ha", "phosphorus_input_kg_ha", "potassium_input_kg_ha", "fertilizer_dose_kg_ha", "pesticide_use_litre_ha", "seed_grade_score"],
    "Production/performance variables": ["harvest_yield_t_ha", "total_harvest_tonnes", "water_productivity_t_1000m3"],
    "Economic variables": ["selling_price_inr_tonne", "cultivation_cost_inr", "gross_income_inr", "net_return_inr"],
    "Risk/resource variables": ["irrigation_volume_m3", "crop_health_risk_pct"]
}
for group, cols in variable_groups.items():
    print(f"{group}: {cols}")


## 7. Statistical Analysis


In [ ]:
# Fresh label: Expanded numeric summary
stats_table = analysis_data[numeric_features].describe().T
stats_table["range"] = stats_table["max"] - stats_table["min"]
stats_table["iqr"] = stats_table["75%"] - stats_table["25%"]
stats_table


In [ ]:
# Fresh label: Mean, middle value and spread table
statistical_summary = pd.DataFrame({
    "Average": analysis_data[numeric_features].mean(),
    "Middle Value": analysis_data[numeric_features].median(),
    "Standard Spread": analysis_data[numeric_features].std(),
    "Smallest": analysis_data[numeric_features].min(),
    "Largest": analysis_data[numeric_features].max()
})
statistical_summary


In [ ]:
# Fresh label: Season-wise numeric behaviour
season_summary = analysis_data.groupby("growing_season")[numeric_features].agg(["mean", "median", "std"]).round(2)
season_summary


## 8. Univariate Analysis


In [ ]:
# Fresh label: Season record balance
plt.figure(figsize=(7, 5))
sns.countplot(data=analysis_data, x="growing_season")
plt.title("Record Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()


In [ ]:
# Fresh label: Crop representation
plt.figure(figsize=(10, 5))
sns.countplot(data=analysis_data, x="crop_type", order=analysis_data["crop_type"].value_counts().index)
plt.xticks(rotation=45, ha="right")
plt.title("Crop Representation In Dataset")
plt.xlabel("Crop")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()


In [ ]:
# Fresh label: Output rate distribution
plt.figure(figsize=(8, 5))
sns.histplot(data=analysis_data, x="harvest_yield_t_ha", kde=True)
plt.title("Harvest Yield Distribution")
plt.xlabel("Harvest yield per hectare")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Fresh label: Earning result distribution
plt.figure(figsize=(8, 5))
sns.histplot(data=analysis_data, x="net_return_inr", kde=True)
plt.title("Net Return Distribution")
plt.xlabel("Net return")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Fresh label: Rainfall distribution
plt.figure(figsize=(8, 5))
sns.histplot(data=analysis_data, x="rainfall_level_mm", kde=True)
plt.title("Rainfall Level Distribution")
plt.xlabel("Rainfall level")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 9. Outlier Analysis


In [ ]:
# Fresh label: IQR unusual-record table
outlier_rows = []
for col in numeric_features:
    q1 = analysis_data[col].quantile(0.25)
    q3 = analysis_data[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((analysis_data[col] < lower) | (analysis_data[col] > upper)).sum()
    outlier_rows.append({"Feature": col, "Lower Fence": lower, "Upper Fence": upper, "Unusual Count": count, "Unusual Percent": round(count / len(analysis_data) * 100, 2)})
outlier_table = pd.DataFrame(outlier_rows).sort_values("Unusual Count", ascending=False)
outlier_table


In [ ]:
# Fresh label: Yield unusual-value visual
plt.figure(figsize=(8, 5))
sns.boxplot(data=analysis_data, y="harvest_yield_t_ha")
plt.title("Harvest Yield Boxplot")
plt.ylabel("Harvest yield per hectare")
plt.tight_layout()
plt.show()


### Univariate: Season distribution (Pie Chart)


In [ ]:
# Fresh label: Seasonal share view
plt.figure(figsize=(7, 7))
analysis_data["growing_season"].value_counts().plot.pie(autopct="%1.1f%%", startangle=90, cmap="viridis")
plt.title("Season Share In Dataset")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 10. Bivariate Analysis


In [ ]:
# Fresh label: Season and yield spread
plt.figure(figsize=(8, 5))
sns.boxplot(data=analysis_data, x="growing_season", y="harvest_yield_t_ha")
plt.title("Yield Spread Across Seasons")
plt.xlabel("Season")
plt.ylabel("Harvest yield")
plt.tight_layout()
plt.show()


### Bar plots for Bivariate Analysis


In [ ]:
# Fresh label: Average yield by season
plt.figure(figsize=(8, 5))
sns.barplot(data=analysis_data, x="growing_season", y="harvest_yield_t_ha", errorbar=None)
plt.title("Average Harvest Yield Across Seasons")
plt.xlabel("Season")
plt.ylabel("Average yield")
plt.tight_layout()
plt.show()


In [ ]:
# Fresh label: Average net return by season
plt.figure(figsize=(8, 5))
sns.barplot(data=analysis_data, x="growing_season", y="net_return_inr", errorbar=None)
plt.title("Average Net Return Across Seasons")
plt.xlabel("Season")
plt.ylabel("Average net return")
plt.tight_layout()
plt.show()


In [ ]:
# Fresh label: Rainfall and output rate
plt.figure(figsize=(8, 5))
sns.scatterplot(data=analysis_data, x="rainfall_level_mm", y="harvest_yield_t_ha", alpha=0.55)
plt.title("Rainfall Level Versus Harvest Yield")
plt.xlabel("Rainfall level")
plt.ylabel("Harvest yield")
plt.tight_layout()
plt.show()


## 11. Multivariate Analysis


In [ ]:
# Fresh label: Crop, season and yield
plt.figure(figsize=(12, 6))
sns.boxplot(data=analysis_data, x="crop_type", y="harvest_yield_t_ha", hue="growing_season")
plt.xticks(rotation=45, ha="right")
plt.title("Yield By Crop And Season")
plt.xlabel("Crop")
plt.ylabel("Harvest yield")
plt.legend(title="Season")
plt.tight_layout()
plt.show()


### Multivariate: Grouped Bar Plot of Average Yield by Crop and Season


In [ ]:
# Fresh label: Crop-season output table
crop_season_summary = (analysis_data.groupby(["crop_type", "growing_season"]).agg(Average_Yield=("harvest_yield_t_ha", "mean"), Average_Net_Return=("net_return_inr", "mean"), Average_Water_Productivity=("water_productivity_t_1000m3", "mean")).round(2).reset_index())
plt.figure(figsize=(14, 7))
sns.barplot(data=crop_season_summary, x="crop_type", y="Average_Yield", hue="growing_season", palette="viridis")
plt.title("Average Yield By Crop And Season")
plt.xlabel("Crop")
plt.ylabel("Average yield")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Season")
plt.tight_layout()
plt.show()
crop_season_summary.sort_values(["growing_season", "Average_Yield"], ascending=[True, False])


### Multivariate: Pair Plot of Key Environmental and Performance Metrics by Season


In [ ]:
# Fresh label: Weather-performance relationship grid
selected_columns = ["mean_temp_c", "rainfall_level_mm", "relative_humidity_pct", "harvest_yield_t_ha", "net_return_inr", "growing_season"]
sns.pairplot(analysis_data[selected_columns], hue="growing_season", palette="viridis", plot_kws={"alpha": 0.6})
plt.suptitle("Environment And Performance Metrics By Season", y=1.02)
plt.show()


In [ ]:
# Fresh label: Numerical relationship heatmap
plt.figure(figsize=(16, 12))
corr = analysis_data[numeric_features].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Matrix Of Notebook Feature Aliases")
plt.tight_layout()
plt.show()


## 12. Seasonal Comparison


In [ ]:
# Fresh label: Seasonal performance pattern table
season_metrics = analysis_data.groupby("growing_season").agg(
    Records=("farm_code", "count"), Average_Yield=("harvest_yield_t_ha", "mean"), Total_Production=("total_harvest_tonnes", "sum"),
    Average_Net_Return=("net_return_inr", "mean"), Total_Net_Return=("net_return_inr", "sum"), Average_Water_Used=("irrigation_volume_m3", "mean"),
    Average_Water_Productivity=("water_productivity_t_1000m3", "mean"), Average_Risk=("crop_health_risk_pct", "mean"),
    Average_Rainfall=("rainfall_level_mm", "mean"), Average_Temperature=("mean_temp_c", "mean")
).round(2)
season_metrics


In [ ]:
# Fresh label: Crop performance within seasons
crop_season_performance = analysis_data.groupby(["growing_season", "crop_type"]).agg(Average_Yield=("harvest_yield_t_ha", "mean"), Average_Net_Return=("net_return_inr", "mean"), Average_Water_Efficiency=("water_productivity_t_1000m3", "mean")).round(2).sort_values(["growing_season", "Average_Yield"], ascending=[True, False])
crop_season_performance


## 13. Additional Student-Driven Analysis


In [ ]:
# Student Analysis 1 - Regional seasonal view
state_season_view = analysis_data.groupby(["state_name", "growing_season"]).agg(Records=("farm_code", "count"), Average_Yield=("harvest_yield_t_ha", "mean"), Average_Net_Return=("net_return_inr", "mean")).round(2).reset_index()
plt.figure(figsize=(14, 6))
sns.barplot(data=state_season_view, x="state_name", y="Average_Yield", hue="growing_season")
plt.title("State-Level Yield Across Seasons")
plt.xlabel("State")
plt.ylabel("Average yield")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
state_season_view.sort_values("Average_Yield", ascending=False).head(15)


In [ ]:
# Student Analysis 2 - Watering system efficiency
watering_season_view = analysis_data.groupby(["watering_system", "growing_season"]).agg(Average_Yield=("harvest_yield_t_ha", "mean"), Average_Water_Productivity=("water_productivity_t_1000m3", "mean"), Average_Net_Return=("net_return_inr", "mean")).round(2).reset_index()
plt.figure(figsize=(12, 6))
sns.barplot(data=watering_season_view, x="watering_system", y="Average_Water_Productivity", hue="growing_season")
plt.title("Water Productivity By Irrigation Method And Season")
plt.xlabel("Watering system")
plt.ylabel("Average water productivity")
plt.tight_layout()
plt.show()
watering_season_view.sort_values("Average_Water_Productivity", ascending=False)


In [ ]:
# Student Analysis 3 - Crop health risk and returns
risk_band = pd.cut(analysis_data["crop_health_risk_pct"], bins=[0, 35, 50, 100], labels=["Lower risk", "Medium risk", "Higher risk"])
risk_view_data = analysis_data.assign(risk_band=risk_band)
risk_season_view = risk_view_data.groupby(["growing_season", "risk_band"], observed=False).agg(Records=("farm_code", "count"), Average_Yield=("harvest_yield_t_ha", "mean"), Average_Net_Return=("net_return_inr", "mean")).round(2).reset_index()
plt.figure(figsize=(10, 5))
sns.barplot(data=risk_season_view, x="risk_band", y="Average_Net_Return", hue="growing_season")
plt.title("Net Return By Risk Band And Season")
plt.xlabel("Crop health risk band")
plt.ylabel("Average net return")
plt.tight_layout()
plt.show()
risk_season_view


## 14. Key Insights

1. Kharif has the highest average yield in the seasonal comparison table.
2. Zaid shows weaker average economic returns than Kharif and Rabi.
3. Crop type strongly changes yield levels.
4. Watering systems show different yield and water productivity patterns.
5. Yield, production and net return are positively related.
6. Rainfall differs noticeably by season.
7. Some numeric features contain unusual values.
8. Risk-band analysis shows profit should be interpreted with crop health risk.


## 15. Recommendations

- Give more attention to Kharif practices because this dataset shows stronger average yield and net return in that season.
- Review Zaid-season records for cost control, crop selection and market price improvement.
- Compare crop performance within each season before crop planning decisions.
- Study watering-system performance further because yield and water productivity differ by method.
- Use risk-band analysis while planning because pest/disease risk can affect seasonal interpretation.


## 16. Conclusion

The dataset reveals clear season-wise differences in agricultural performance. Kharif generally appears stronger for yield and net return, while Zaid needs closer economic review. Crop type, irrigation method, water productivity, rainfall and risk all add important context to seasonal comparison.

The analysis is descriptive, so it supports evidence-based discussion and future investigation, but it should not be treated as proof of cause and effect.


## 17. Seasonal Performance Patterns

Kharif records show stronger average yield and net return, Rabi stays closer to the middle, and Zaid appears weaker in economic performance. Rainfall, temperature, water use, crop type and risk also differ across seasons, so seasonal performance should be interpreted with these supporting factors.


In [ ]:
# Fresh label: Season pattern ranking
season_pattern_rank = season_metrics.sort_values("Average_Yield", ascending=False)
season_pattern_rank


## 18. Most Important Findings, Limitations and Future Decision-Making Usefulness

### Most important findings

- Kharif performs best on average yield and net return.
- Zaid has lower average profit performance.
- Crop type creates major yield differences.
- Water productivity and yield are strongly connected.
- Irrigation method is important for comparing efficiency and output.

### Limitations

- This is descriptive analysis, not causal proof.
- Missing values were treated inside the notebook using simple rules.
- The dataset sample may not represent all farms.
- Crop, region, farm size and irrigation method can affect seasonal comparisons.
- Unusual values need domain review before removal.

### Future decision-making me usefulness

This work can help identify strong and weak seasons, compare crop-season combinations, review irrigation efficiency, detect risk-sensitive areas and decide where deeper farm planning or additional data collection is needed.


## 19. Project Checklist

- [x] Dataset loaded successfully
- [x] Top 5 rows analyzed
- [x] Dataset shape and structure examined
- [x] Data types examined
- [x] Missing values identified and handled
- [x] Duplicate records identified and handled
- [x] Descriptive/statistical analysis performed
- [x] Outliers investigated
- [x] Univariate analysis completed
- [x] Bivariate analysis completed
- [x] Multivariate analysis completed
- [x] Correlation analysis completed
- [x] Seasonal comparisons performed
- [x] At least 3 student-designed analyses completed
- [x] At least 8 meaningful insights documented
- [x] Evidence-based recommendations provided
- [x] Limitations discussed
- [x] Final conclusion provided
